In [ ]:
import waveorder as wo
from waveorder.waveorder_reconstructor import waveorder_microscopy as setup
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
from waveorder.visual import plotVectorField
import zarr

In [ ]:
def load_bg(bg_path, height, width):
    bg_paths = glob.glob(bg_path+'*.tif')
    bg_paths.sort()
#     print(bg_paths)
    bg_data = np.zeros([len(bg_paths),height,width])
    
    for i in range(len(bg_paths)):
        bg_data[i,:,:] = tiff.imread(bg_paths[i])

    return bg_data

In [ ]:
def add_colorbar(mappable):
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    import matplotlib.pyplot as plt
    last_axes = plt.gca()
    ax = mappable.axes
    fig = ax.figure
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(mappable, cax=cax)
    plt.sca(last_axes)
    return cbar

def read_inst_mat(path):
    with open(path, 'r') as meta:
        file = json.load(meta)
    
    inst_mat = np.asarray(file['Summary']['Instrument Matrix'])
    
    return inst_mat

In [ ]:
path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV_IFN/Coverslip_1/C1_MultiChan_Stack_1/'

tiff_paths = glob.glob(path+'*.tif')
tiff_paths.sort()


In [ ]:
with tiff.TiffFile(tiff_paths[0]) as tif:
    mm_meta = tif.micromanager_metadata

    if mm_meta['Summary']['Positions'] > 1:
        stage_positions = []
        for p in range(mm_meta['Summary']['Positions']):
            print(f"appending position: {mm_meta['Summary']['StagePositions'][p]['Label']}")
            stage_positions.append(mm_meta['Summary']['StagePositions'][p]['Label'])

    chnames = []
    for ch in mm_meta['Summary']['ChNames']:
        print(f"appending {ch}")
        chnames.append(ch)

    height = mm_meta['Summary']['Height']
    width = mm_meta['Summary']['Width']
    frames = mm_meta['Summary']['Frames']
    slices = mm_meta['Summary']['Slices']
    channels = mm_meta['Summary']['Channels']

In [ ]:
positions = {}
with tiff.TiffFile(tiff_paths[0]) as tif:
    for idx, tiffpageseries in enumerate(tif.series):
        print(idx, tiffpageseries)
        positions[idx] = zarr.open(tiffpageseries.aszarr(), mode='r')

In [ ]:
img_dim = (height,width)
lambda_illu   = 0.525                         # illumination wavelength (um)
mag           = 40                           # magnification of the microscope                      
NA_obj        = 1.2            # detection NA of the objective
n_media       = 1.33 
NA_illu       = 0.4                           # illumination NA of the condenser
N_defocus     = 65                            # number of defocus images
N_channel     = 4                             # number of Polscope channels
N_pattern     = 1                             # number of illumination patterns
z_step        = 0.25                           # z_step of the stack
z_defocus     = (np.r_[:N_defocus]-N_defocus//2)*z_step  # z positions of the stack
ps            = 6.45/mag                    # effective pixel size at the sample plane (cam pix/mag in um)
cali          = True                         # correction for S1/S2 Polscope reconstruction (does not affect phase)
bg_option     = 'global'                      # background correction method for Polscope recon (does not affect phase)
use_gpu       = False                          # option to use gpu or not (required cupy)
gpu_id        = 0                             # id of gpu to use

chi = 0.05
inst_mat = np.array([[1, 0, 0, -1],
                     [1, np.sin(2 * np.pi * chi), 0, -np.cos(2 * np.pi * chi)],
                     [1, -0.5 * np.sin(2 * np.pi * chi), np.sqrt(3) * np.cos(np.pi * chi) * np.sin(np.pi * chi), -np.cos(2 * np.pi * chi)],
                     [1, -0.5 * np.sin(2 * np.pi * chi), -np.sqrt(3) / 2 * np.sin(2 * np.pi * chi), -np.cos(2 * np.pi * chi)]])

# A_matrix = read_inst_mat(meta_path)
print(f'Instrument Matrix: \n\n{inst_mat}')

In [ ]:
np.max(recon_data[1])

In [ ]:
recon_data[1]

In [ ]:
%%time
recon = setup(img_dim, lambda_illu, ps, NA_obj, NA_illu, z_defocus, chi=None,
                 n_media=n_media, cali=cali, bg_option=bg_option, 
                 A_matrix=inst_mat, QLIPP_birefringence_only = False, 
                 phase_deconv='3D',illu_mode='BF', use_gpu=False, gpu_id=0)

bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/2021_02_03_40x_04NA_A549/48hr_RSV_IFN/BG/'
bg_data = load_bg(bg_path, height, width)

bg_stokes = recon.Stokes_recon(bg_data)
bg_stokes = recon.Stokes_transform(bg_stokes)

In [ ]:
np.min(bg_stokes[0])

In [ ]:
%%time
for pos in range(1):
    S0_stack = np.zeros([positions[pos].shape[0], positions[pos].shape[-2],positions[pos].shape[-1]])
    retardance_stack = np.zeros([positions[pos].shape[0], positions[pos].shape[-2],positions[pos].shape[-1]])
    for z in range(positions[0].shape[0]):
        stokes_data = recon.Stokes_recon(positions[pos][z][0:4])
        stokes_data = recon.Stokes_transform(stokes_data)
        S_image_tm = recon.Polscope_bg_correction(stokes_data, bg_stokes)
        
        S0_stack[z,:,:] = S_image_tm[0]
        recon_data = recon.Polarization_recon(S_image_tm)
        retardance_stack[z,:,:] = recon_data[0] / (2 * np.pi) * lambda_illu*1000
        
    S0_stack = np.transpose(S0_stack,(1,2,0))
    phase3d =  recon.Phase_recon_3D(S0_stack, absorption_ratio=0.0, method='Tikhonov', reg_re = 1e-4, reg_im = 1e-4,\
                       rho = 1e-5, lambda_re = 1e-3, lambda_im = 1e-3, itr = 20, verbose=True)

In [ ]:
positions_test = [positions[0], positions[1]]

In [ ]:
positions

In [ ]:
positions = [positions[0], positions[1]]

In [ ]:
wo.visual.image_stack_viewer_fast(retardance_stack, vrange=(0,20))

In [ ]:
wo.visual.image_stack_viewer_fast(np.transpose(phase3d,(2,1,0)))

In [ ]:
plt.figure(dpi=300)
plt.imshow(phase3d[:,:,30], 'gray')

In [ ]:
def init_zarr_store(data_path, data_shape, data_structure):
    chunk_size = (1, 65, 1) + data_shape[-2:] # Dimension order PZCYX
    compressor=Blosc(cname='zstd', clevel=1, shuffle=Blosc.SHUFFLE)
    
    if not data_path.endswith('.zarr'):
        data_path += '.zarr'
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
        
    zarr_store = zarr.open(data_path, mode='a')
    
    for group, slips, datasets in data_structure.items():
        zarr_store.create_group(group)
        for ds in datasets:
#             synchronizer = zarr.ProcessSynchronizer(os.path.join(data_path, group, ds)+'.sync')
            zarr_store[group].zeros(ds, shape=data_shape, chunks=chunk_size, dtype='float32',
                                    compressor=compressor)
            
    return zarr_store

In [ ]:
data_structure_ = {'24hr_RSV': {'C1': ['3D_Phase','Retardance', 'Orientation', 'BF']}}
#                   '24hr_RSV_IFN': ['3D_Phase','Retardance', 'Orientation', 'BF'],
#                   '24hr_Mock': ['3D_Phase','Retardance', 'Orientation', 'BF'],
#                   '24hr_Mock_IFN': ['3D_Phase','Retardance', 'Orientation', 'BF']}

In [ ]:
for group, datasets in data_structure_.items():
    print(group, datasets)

In [ ]:
data_structure_.items()